In [ ]:
import pandas as pd
from geopy.geocoders import Nominatim
import time

df = pd.read_csv("../../Data/Processed/agencias_tratado.csv")

df = df.head(10)

geolocator = Nominatim(user_agent="meu_geocoder")

def get_coords(endereco):
    try:
        location = geolocator.geocode(endereco)                                                                                                  
        if location:
            return location.latitude, location.longitude
    except:
        return None, None
    return None, None

latitudes = []
longitudes = []

for i, row in df.iterrows():
    endereco_completo = f"{row['endereco']}, {row['bairro']}, {row['municipio']}, {row['cep']}"
    lat, lon = get_coords(endereco_completo)
    latitudes.append(lat)
    longitudes.append(lon)
    time.sleep(1) 

df["latitude"] = latitudes
df["longitude"] = longitudes

df.to_csv("dados_com_coordenadas.csv", index=False)

In [ ]:
import pandas as pd

df = pd.read_csv("dados_com_coordenadas.csv")


print(df.head())


print("\n Estatísticas ")
print(f"Total de linhas: {len(df)}")
print(f"Latitudes válidas: {df['latitude'].notna().sum()}")
print(f"Longitudes válidas: {df['longitude'].notna().sum()}")

# Ver alguns endereços
print("\n--- Exemplos de endereços ---")
print(df[['endereco', 'bairro', 'municipio', 'cep', 'latitude', 'longitude']].head(10))

   Unnamed: 0                                   nome_instituicao  \
0           0  BANCO DO BRASIL S.A.                          ...   
1           1  BANCO DO BRASIL S.A.                          ...   
2           2  BANCO DO BRASIL S.A.                          ...   
3           3  BANCO DO BRASIL S.A.                          ...   
4           4  BANCO DO BRASIL S.A.                          ...   

                                        nome_agencia  \
0  MANAUS                                        ...   
1  PRESIDENTE VARGAS                             ...   
2  SANTOS                                        ...   
3  CAMPOS GOYTACAZES                             ...   
4  SALVADOR                                      ...   

                                   endereco      numero  \
0  R.GUILHERME MOREIRA,315                                
1  AV.PRES.VARGAS,248                                     
2  R.QUINZE DE NOVEMBRO,195                               
3  RUA QUARTA DA J

In [8]:
import pandas as pd
import folium
from folium import plugins

df = pd.read_csv("dados_com_coordenadas.csv")
df_clean = df.dropna(subset=['latitude', 'longitude'])

# Criar mapa centrado no Brasil
mapa = folium.Map(
    location=[-15.7801, -47.9292],  # Centro do Brasil
    zoom_start=4,
    tiles='OpenStreetMap'
)

# Adicionar marcadores
for idx, row in df_clean.iterrows():
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=folium.Popup(
            f"""
            <b style='font-size:14px'>{row['municipio']}</b><br>
            <b>Bairro:</b> {row['bairro']}<br>
            <b>Endereço:</b> {row['endereco']}<br>
            <b>CEP:</b> {row['cep']}
            """,
            max_width=300
        ),
        tooltip=f"📍 {row['municipio']} - {row['bairro']}",
        icon=folium.Icon(color='red', icon='bank', prefix='fa')
    ).add_to(mapa)

# Adicionar controle de camadas
folium.LayerControl().add_to(mapa)

# Adicionar minimap
minimap = plugins.MiniMap(toggle_display=True)
mapa.add_child(minimap)

# Salvar
mapa.save('mapa_agencias_interativo.html')
print("✓ Mapa interativo salvo como 'mapa_agencias_interativo.html'")
print("  Abra o arquivo no navegador para visualizar!")

✓ Mapa interativo salvo como 'mapa_agencias_interativo.html'
  Abra o arquivo no navegador para visualizar!


In [ ]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderServiceError
import time

df = pd.read_csv("../../Data/Processed/agencias_tratado.csv")
df = df.head(10)

geolocator = Nominatim(user_agent="meu_geocoder_v2", timeout=10)

def get_coords(endereco, tentativas=3):
    for i in range(tentativas):
        try:
            print(f"  Buscando: {endereco}")
            location = geolocator.geocode(endereco, language='pt')
            if location:
                print(f"  ✓ Encontrado: {location.latitude}, {location.longitude}")
                return location.latitude, location.longitude
            else:
                print(f"  ✗ Não encontrado")
        except (GeocoderTimedOut, GeocoderServiceError) as e:
            print(f"  Erro (tentativa {i+1}/{tentativas}): {e}")
            time.sleep(2)
        except Exception as e:
            print(f"  Erro inesperado: {e}")
            break
    return None, None

latitudes = []
longitudes = []
for i, row in df.iterrows():
    endereco_completo = f"{row['endereco']}, {row['bairro']}, {row['municipio']}, {row['cep']}"
    lat, lon = get_coords(endereco_completo)
    latitudes.append(lat)
    longitudes.append(lon)
    time.sleep(1)



[1/10]
  Buscando: R.GUILHERME MOREIRA,315                 , CENTRO                        , MANAUS                                                      , Brasil
  ✓ Encontrado: -3.1352388, -60.0229303

[2/10]
  Buscando: AV.PRES.VARGAS,248                      , CAMPINA                       , BELEM                                                       , Brasil
  ✗ Não encontrado
  Buscando: AV.PRES.VARGAS,248                      , CAMPINA                       , BELEM                                                       , Brasil
  ✗ Não encontrado
  Buscando: AV.PRES.VARGAS,248                      , CAMPINA                       , BELEM                                                       , Brasil
  ✗ Não encontrado
  Buscando: AV.PRES.VARGAS,248                      , BELEM                                                       , Brasil
  ✗ Não encontrado
  Buscando: AV.PRES.VARGAS,248                      , BELEM                                                       , Brasil
  

PermissionError: [Errno 13] Permission denied: 'dados_com_coordenadas.csv'

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Carregar os dados
df = pd.read_csv("dados_com_coordenadas.csv")
df_clean = df.dropna(subset=['latitude', 'longitude'])

print(f"✓ Pontos válidos: {len(df_clean)}/{len(df)}")

# Criar o mapa
fig, ax = plt.subplots(figsize=(16, 12))

# Plot dos pontos
scatter = ax.scatter(df_clean['longitude'], df_clean['latitude'], 
                     c='red', s=300, alpha=0.7, 
                     edgecolors='darkred', linewidth=2, zorder=5)

# Adicionar labels
for idx, row in df_clean.iterrows():
    ax.annotate(f"{row['municipio']}\n{row['bairro']}", 
                (row['longitude'], row['latitude']),
                fontsize=9, fontweight='bold',
                ha='center', va='bottom',
                xytext=(0, 8), textcoords='offset points',
                bbox=dict(boxstyle='round,pad=0.5', facecolor='yellow', alpha=0.7))

# Configurações do gráfico
ax.set_xlabel('Longitude', fontsize=14, fontweight='bold')
ax.set_ylabel('Latitude', fontsize=14, fontweight='bold')
ax.set_title('Localização das Agências Bancárias no Brasil', 
             fontsize=18, fontweight='bold', pad=20)
ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.5)

# Adicionar contorno do "mapa"
ax.spines['top'].set_visible(True)
ax.spines['right'].set_visible(True)
ax.spines['bottom'].set_linewidth(2)
ax.spines['left'].set_linewidth(2)

plt.tight_layout()
plt.savefig('mapa_agencias.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n--- Coordenadas Encontradas ---")
print(df_clean[['municipio', 'bairro', 'endereco', 'latitude', 'longitude']])
